# Axial v3 Iteration B - low-cost runner

Orquestador B0-B6 reproducible. No ejecuta todos los experimentos por defecto.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", ".")).resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [ ]:
from lumbar_mri.axial_v3.experiments import estimate_run_count, load_low_cost_grid, select_experiment
from lumbar_mri.axial_v3.guards import require_train_val_only
from lumbar_mri.axial_v3.training import AxialV3TrainConfig, run_calibration, run_preflight, run_training, summarize_validation_runs

ALLOWED_SPLITS = require_train_val_only(["train", "val"], context="notebook_52")
CONFIG_PATH = REPO_ROOT / "config" / "axial_v3_low_cost_grid.json"
GRID = load_low_cost_grid(CONFIG_PATH)
RUN_MODE = os.getenv("RUN_MODE", "preflight")
EXPERIMENT_ID = os.getenv("PFI_AXIAL_V3_EXPERIMENT_ID", "B0")
EXPLICIT_EXPERIMENT_ID = "PFI_AXIAL_V3_EXPERIMENT_ID" in os.environ

In [ ]:
def train_config_for_experiment(experiment_id):
    base = AxialV3TrainConfig.from_env()
    experiment = select_experiment(GRID, experiment_id)
    return replace(
        base,
        RUN_ID=experiment.runId,
        EXPERIMENT_ID=experiment.experimentId,
        EXPERIMENT_TYPE=experiment.experimentType,
        RAW0_WEIGHT_BOOST=experiment.raw0EffectiveWeightMultiplier,
        MAX_CLASS_WEIGHT_RATIO=experiment.maxClassWeightRatio,
        LOSS_NAME=experiment.lossName,
        TVERSKY_ALPHA=experiment.tverskyAlpha or base.TVERSKY_ALPHA,
        TVERSKY_BETA=experiment.tverskyBeta or base.TVERSKY_BETA,
        FOCAL_GAMMA=experiment.focalGamma or base.FOCAL_GAMMA,
        RAW0_TVERSKY_WEIGHT=experiment.raw0TverskyWeight,
        RAW0_FP_PENALTY_WEIGHT=experiment.raw0FpPenaltyWeight,
        MIN_PROBABILITY=experiment.minProbability,
        MIN_MARGIN=experiment.minMargin,
        MAX_EPOCHS=experiment.maxEpochs,
        EARLY_STOPPING_PATIENCE=experiment.earlyStoppingPatience,
        PRESENCE_HEAD_ENABLED=experiment.presenceHeadEnabled,
        LAMBDA_PRESENCE=experiment.lambdaPresence,
        RAW0_BALANCED_SAMPLER_ENABLED=experiment.raw0BalancedSamplerEnabled,
        POSITIVE_FRACTION=experiment.positiveFraction or 0.5,
    )


CFG = train_config_for_experiment(EXPERIMENT_ID)

In [ ]:
RUN_EXPERIMENT = os.getenv("PFI_RUN_AXIAL_V3_EXPERIMENT", "0") == "1"

if RUN_MODE == "preflight":
    report = run_preflight(CFG, select_experiment(GRID, EXPERIMENT_ID))
    print(json.dumps({"mode": RUN_MODE, "experimentId": EXPERIMENT_ID, "plannedRuns": estimate_run_count(GRID), "preflight": report}, indent=2, default=str))
elif RUN_MODE in {"smoke", "train"}:
    if not EXPLICIT_EXPERIMENT_ID or not RUN_EXPERIMENT:
        raise RuntimeError("smoke/train requieren PFI_AXIAL_V3_EXPERIMENT_ID y PFI_RUN_AXIAL_V3_EXPERIMENT=1")
    print(json.dumps(run_training(CFG, smoke=RUN_MODE == "smoke"), indent=2, default=str))
elif RUN_MODE == "calibrate":
    if not EXPLICIT_EXPERIMENT_ID or not RUN_EXPERIMENT:
        raise RuntimeError("calibrate requiere PFI_AXIAL_V3_EXPERIMENT_ID y PFI_RUN_AXIAL_V3_EXPERIMENT=1")
    parent = Path(os.getenv("PFI_AXIAL_V3_PARENT_CHECKPOINT", ""))
    print(json.dumps(run_calibration(CFG, parent_checkpoint_path=parent), indent=2, default=str))
elif RUN_MODE == "summarize":
    print(json.dumps(summarize_validation_runs(CFG), indent=2, default=str))
else:
    raise ValueError(f"RUN_MODE invalido: {RUN_MODE}")